# Notebook 01 — Exploração do Site do Diário Oficial de Avaré

**Etapa 1 — Semana 1**

Este notebook documenta interativamente a exploração do site `imprensaoficialmunicipal.com.br/avare` e a execução do coletor automatizado.

---

In [ ]:
# Instalar dependências (caso ainda não estejam instaladas)
# !pip install requests beautifulsoup4 pandas

## 1. Requisição Básica ao Site

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = 'https://imprensaoficialmunicipal.com.br/avare'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'pt-BR,pt;q=0.9'
}

response = requests.get(URL, headers=headers, timeout=15)
print(f'Status HTTP: {response.status_code}')
print(f'Tamanho do conteúdo: {len(response.text):,} caracteres')
print(f'Encoding detectado: {response.encoding}')

> ⚠️ **Nota:** Se o status for 403, você está em uma rede de datacenter (nuvem).
> Execute este notebook a partir da sua rede local.

## 2. Inspecionar Estrutura HTML

In [ ]:
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    print('Título:', soup.title.string if soup.title else 'Sem título')
    print()
    
    links = soup.find_all('a', href=True)
    print(f'Total de links encontrados: {len(links)}')
    print('\nPrimeiros 20 links:')
    for link in links[:20]:
        print(f"  {link['href'][:60]:<60} | {link.get_text(strip=True)[:40]}")

## 3. Explorar Seções do Diário Oficial

In [ ]:
if response.status_code == 200:
    # Extrai itens da lista lateral (seções com contagem)
    secoes = []
    for li in soup.find_all('li'):
        txt = li.get_text(strip=True)
        if '(' in txt and ')' in txt:
            try:
                contagem = int(txt.split('(')[1].split(')')[0])
                nome = txt.split(') ')[1] if ') ' in txt else txt
                secoes.append({'secao': nome, 'total_atos': contagem})
            except:
                pass
    
    if secoes:
        df_secoes = pd.DataFrame(secoes).sort_values('total_atos', ascending=False)
        print('Seções encontradas:')
        display(df_secoes)

## 4. Coletar Lista de Portarias

In [ ]:
URL_PORTARIAS = 'https://imprensaoficialmunicipal.com.br/listaatos.php?c=Avaré&s=Portarias'

resp_portarias = requests.get(URL_PORTARIAS, headers=headers, timeout=15)
print(f'Status: {resp_portarias.status_code} | Tamanho: {len(resp_portarias.text):,} chars')

if resp_portarias.status_code == 200:
    soup2 = BeautifulSoup(resp_portarias.text, 'html.parser')
    tabela = soup2.find('table')
    
    if tabela:
        linhas = tabela.find_all('tr')[1:]  # pula cabeçalho
        print(f'\nTotal de linhas na tabela: {len(linhas)}')
        
        publicacoes = []
        for linha in linhas:
            cols = linha.find_all('td')
            if len(cols) >= 5:
                link_ed  = cols[4].find('a')
                link_txt = cols[5].find('a') if len(cols) > 5 else None
                publicacoes.append({
                    'titulo_ato': cols[0].get_text(strip=True),
                    'data_publicacao': cols[1].get_text(strip=True),
                    'numero_edicao': cols[2].get_text(strip=True),
                    'tipo_ato': 'Portarias',
                    'url_documento': link_ed['href'] if link_ed else '',
                    'url_texto': link_txt['href'] if link_txt else '',
                })
        
        df_portarias = pd.DataFrame(publicacoes)
        display(df_portarias.head(10))
    else:
        print('Nenhuma tabela encontrada na página.')

## 5. Executar o Scraper Completo

In [ ]:
import sys
sys.path.append('..')  # adiciona o diretório raiz ao path

from src.scraper import executar_coleta

df = executar_coleta()

if not df.empty:
    print(f'\nTotal coletado: {len(df)} publicações')
    display(df.describe(include='all'))

## 6. Análise Exploratória do CSV Gerado

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv('../data/diario_avare.csv')
print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
display(df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição por tipo de ato
df['tipo_ato'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Publicações por Tipo de Ato')
axes[0].set_xlabel('Tipo')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

# Distribuição por secretaria
df['secretaria'].value_counts().plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Publicações por Secretaria (inferido)')
axes[1].set_xlabel('Quantidade')

plt.tight_layout()
plt.savefig('../data/processed/distribuicao_publicacoes.png', dpi=150)
plt.show()
print('Gráfico salvo!')

---
## ✅ Conclusão da Etapa 1

- Site mapeado com sucesso
- Coletor funcionando via `requests + BeautifulSoup` em rede local
- CSV gerado em `data/diario_avare.csv`
- Próximo passo: **Etapa 2** — executar `python src/extract_text.py`